# Challenge W4: NLP - Fake News


Antonio Traquinas
## MODEL_AT_2


<br>

<hr>

## Steps:
    1. Import data
    2. EDA
    3. Data Splitting
    4. Text Preprocessing 
    5. Feature Extraction
    6. Model Training

##  Libraries:

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
#import seaborn as sns

import re
import string

from sklearn.model_selection import train_test_split

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# 1. Load data

In [2]:
df = pd.read_csv("./dataset/data.csv")

In [3]:
df.shape
df.head()       
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39942 entries, 0 to 39941
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   label    39942 non-null  int64
 1   title    39942 non-null  str  
 2   text     39942 non-null  str  
 3   subject  39942 non-null  str  
 4   date     39942 non-null  str  
dtypes: int64(1), str(4)
memory usage: 1.5 MB


# 2. EDA

In [4]:
df.isnull().sum() # missing values?

label      0
title      0
text       0
subject    0
date       0
dtype: int64

In [5]:
print("Duplicate rows:", df.duplicated().sum()) # Duplicates?


Duplicate rows: 201


In [6]:
df.sample(5, random_state=42)

,label,title,text,subject,date
6524,1,Oil business seen in strong position as Trump ...,(This January 3 story was corrected to remove...,politicsNews,"January 3, 2017"
30902,0,WHOA! COLLEGE SNOWFLAKE FREAKS OUT: Screams Fo...,So much for healthy debate on college campus I...,politics,"May 12, 2017"
36459,0,CRONY CORRUPT POLITICS: Obama Admin BLOCKED FB...,The information is spilling out little by litt...,Government News,"Aug 11, 2016"
9801,1,Cruz campaign vetting Fiorina as a possible VP...,WASHINGTON (Reuters) - U.S. Republican preside...,politicsNews,"April 25, 2016"
25638,0,Minnesota Woman Writes Amazing F*ck Off Lette...,"Attention, conservative men. This one is for y...",News,"July 2, 2016"


In [7]:
print("Unique subjects:", df['subject'].nunique())
df['subject'].value_counts()

Unique subjects: 6


subject
politicsNews       11272
News                9050
worldnews           8727
politics            6841
left-news           2482
Government News     1570
Name: count, dtype: int64

In [8]:
pd.crosstab(df['subject'], df['label']) # Because subject and label are correlated we should drop "subject" column to avoid data leakage.

label,0,1
subject,,
Government News,1570,0
News,9050,0
left-news,2482,0
politics,6841,0
politicsNews,0,11272
worldnews,0,8727


# 3. TRAIN TEST VALIDATION SPLIT

In [9]:
# Dropping columns subject and date as the have been shown to not carry important information
df = df.drop(columns=['subject','date']) 

In [10]:
df = df.drop_duplicates() # Dropping 201 duplicates that appeared once subject and date were removed (meaning that they had same title and/or text and we 
                            #could fall into data leakage)
print(df.shape) 

(36429, 3)


In [11]:

# Split into features and target
# Adjust the target column name to match your dataset (e.g. 'label', 'class', 'Spam/Ham')
X = df['title'] + ' ' + df['text']   # <-- replace 'label' with your actual target column
y = df['label']

# First split: 70% train, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: Split the remaining 30% into 15% validation and 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

# Display dataset shapes
print(f"Train shape:      {X_train.shape}")
print(f"Validation shape: {X_val.shape}")
print(f"Test shape:       {X_test.shape}")

# Display class balance
print("\nTrain class balance:")
print(y_train.value_counts(normalize=True))

print("\nValidation class balance:")
print(y_val.value_counts(normalize=True))

print("\nTest class balance:")
print(y_test.value_counts(normalize=True))

Train shape:      (25500,)
Validation shape: (5464,)
Test shape:       (5465,)

Train class balance:
label
1    0.543216
0    0.456784
Name: proportion, dtype: float64

Validation class balance:
label
1    0.543192
0    0.456808
Name: proportion, dtype: float64

Test class balance:
label
1    0.543275
0    0.456725
Name: proportion, dtype: float64


# 4. Text Preprocessing 
    - we will combine it into a reusable function 

In [12]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower() # Lowercasing
    text = re.sub(r'[^a-z\s]', '', text) # Remove special characters
    tokens = word_tokenize(text) # Tokenization
    tokens = [word for word in tokens if word not in stop_words] # stopword removal
    tokens = [lemmatizer.lemmatize(word) for word in tokens] # Lemmatization
    return ' '.join(tokens)


Applying the text processing to X train, test and val

In [13]:
X_train_clean = X_train.apply(clean_text)
X_test_clean = X_test.apply(clean_text)
X_val_clean = X_val.apply(clean_text)

# 5. Feature extraction - GloVe embeddings

In [14]:
import gensim.downloader as api
import numpy as np

glove = api.load('glove-wiki-gigaword-100')

def document_vector(text):
    words=text.split()
    vectors=[glove[w] for w in words if w in glove]
    if len(vectors)==0:
        return np.zeros(glove.vector_size)
    return np.mean(vectors,axis=0)

X_train_glove=np.vstack(X_train_clean.apply(document_vector))
X_val_glove=np.vstack(X_val_clean.apply(document_vector))
X_test_glove=np.vstack(X_test_clean.apply(document_vector))

print(X_train_glove.shape)


In [15]:
# GloVe document embeddings created above.

(25500, 5000)


# 6. Model training - Linear SVM

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

svm=LinearSVC(C=1.0,random_state=42)
svm.fit(X_train_glove,y_train)

y_val_pred=svm.predict(X_val_glove)

print("=== Linear SVM + GloVe ===")
print("Accuracy:",accuracy_score(y_val,y_val_pred))
print(classification_report(y_val,y_val_pred))
print(confusion_matrix(y_val,y_val_pred))


=== Logistic Regression + TF-IDF (unigrams) ===
Accuracy: 0.984809663250366
              precision    recall  f1-score   support

           0       0.99      0.98      0.98      2496
           1       0.98      0.99      0.99      2968

    accuracy                           0.98      5464
   macro avg       0.99      0.98      0.98      5464
weighted avg       0.98      0.98      0.98      5464

[[2439   57]
 [  26 2942]]


In [ ]:
import joblib

y_test_pred=svm.predict(X_test_glove)
print("Test Accuracy:",accuracy_score(y_test,y_test_pred))

joblib.dump(svm,"linear_svm_glove.pkl")
print("Model saved as linear_svm_glove.pkl")



=== Logistic Regression + TF-IDF (uni+bigrams) ===
Accuracy: 0.9870058565153733
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      2496
           1       0.98      0.99      0.99      2968

    accuracy                           0.99      5464
   macro avg       0.99      0.99      0.99      5464
weighted avg       0.99      0.99      0.99      5464

[[2446   50]
 [  21 2947]]
